# Análise Exploratória de Dados — Dataset de Obesidade

Este notebook realiza a análise exploratória completa do dataset de obesidade utilizado no FIAP Tech Challenge Fase 4.

**Objetivos:**
- Carregar e validar os dados usando os módulos `src/data/validator.py` e `src/data/cleaner.py`
- Gerar estatísticas univariadas para features numéricas
- Analisar distribuições de frequência para features categóricas
- Identificar desbalanceamento nas 7 classes de obesidade
- Visualizar histogramas e box plots das features numéricas
- Calcular e visualizar a correlação de Spearman entre features e o target

**Dataset:** `data/Obesity.csv` — 2111 registros × 17 colunas  
**Requisitos atendidos:** 4.1, 4.2, 4.3, 4.4, 4.5, 13.1

In [ ]:
# Importações
import os
import sys
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Configurações de visualização
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='muted')

# Logging básico para o notebook
logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')
logger = logging.getLogger('eda')

print('Importações concluídas.')

In [ ]:
# Adicionar raiz do projeto ao sys.path para importar src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.validator import DataValidator
from src.data.cleaner import DataCleaner
from src.config import OBESITY_LABELS_PT, NUMERIC_RANGES, CATEGORICAL_VALUES

print(f'Raiz do projeto: {PROJECT_ROOT}')
print('Módulos src/ importados com sucesso.')

In [ ]:
# Carregar, validar e limpar os dados
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'Obesity.csv')

# Carregamento do CSV bruto
df_raw = pd.read_csv(DATA_PATH)
logger.info('CSV carregado. Shape: %s', df_raw.shape)

# Validação com DataValidator
validator = DataValidator()
validation_result = validator.validate(df_raw)

if validation_result.is_valid:
    print('✅ Validação aprovada — nenhum erro encontrado.')
else:
    print(f'⚠️  Validação com {len(validation_result.errors)} aviso(s):')
    for err in validation_result.errors[:10]:
        print(f'   • {err}')

# Limpeza com DataCleaner
cleaner = DataCleaner()
df = cleaner.clean(df_raw)
logger.info('Limpeza concluída. Shape final: %s', df.shape)

print(f'\nShape após limpeza: {df.shape}')
print(f'Duplicatas removidas: {len(df_raw) - len(df)}')
print(f'Valores ausentes restantes: {df.isnull().sum().sum()}')

## 1. Visão Geral do Dataset

Antes de qualquer análise, é importante entender a estrutura do dataset: número de registros, tipos de dados e uma amostra das primeiras linhas.

O dataset contém **17 colunas**: 8 numéricas (Age, Height, Weight, FCVC, NCP, CH2O, FAF, TUE) e 9 categóricas (Gender, family_history, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS, Obesity). A coluna `Obesity` é o **target** com 7 classes.

In [ ]:
# Visão geral do dataset
print('=== Shape ===')
print(f'Linhas: {df.shape[0]:,}  |  Colunas: {df.shape[1]}')

print('\n=== Tipos de dados ===')
print(df.dtypes.to_string())

print('\n=== Primeiras 5 linhas ===')
df.head()

## 2. Estatísticas Univariadas — Features Numéricas

As estatísticas descritivas (média, mediana, desvio padrão, mínimo, máximo e quartis) permitem identificar a distribuição de cada feature numérica e detectar possíveis anomalias.

**Features numéricas analisadas:** Age, Height, Weight, FCVC (frequência de consumo de vegetais), NCP (número de refeições principais), CH2O (consumo de água), FAF (frequência de atividade física), TUE (tempo de uso de tecnologia).

**Implicações:** Valores de FCVC, NCP, CH2O, FAF e TUE são discretizados em escalas de 0–3 ou 1–4, o que explica distribuições não-normais. Age e Weight tendem a ter distribuições assimétricas positivas dado o perfil do dataset.

In [ ]:
# Colunas numéricas (derivadas de NUMERIC_RANGES para evitar hard-code)
NUMERIC_COLS = list(NUMERIC_RANGES.keys())

# Estatísticas descritivas completas
stats = df[NUMERIC_COLS].agg(['mean', 'median', 'std', 'min', 'max']).T
stats.columns = ['Média', 'Mediana', 'Desvio Padrão', 'Mínimo', 'Máximo']

# Adicionar quartis
quartis = df[NUMERIC_COLS].quantile([0.25, 0.50, 0.75]).T
quartis.columns = ['Q1 (25%)', 'Q2 (50%)', 'Q3 (75%)']

stats_completo = pd.concat([stats, quartis], axis=1)
stats_completo = stats_completo.round(4)

print('=== Estatísticas Univariadas — Features Numéricas ===')
stats_completo

## 3. Distribuições de Frequência — Features Categóricas

Para as features categóricas, analisamos a frequência absoluta e relativa de cada categoria. Isso revela possíveis desbalanceamentos nas variáveis preditoras que podem influenciar o modelo.

**Features categóricas analisadas:** Gender, family_history, FAVC, CAEC, SMOKE, SCC, CALC, MTRANS (excluindo Obesity, que é analisada separadamente na seção 4).

**Implicações:** Desbalanceamentos em features como SMOKE ou SCC podem indicar que o modelo terá dificuldade em aprender padrões associados às categorias minoritárias.

In [ ]:
# Features categóricas (excluindo Obesity — analisada na seção 4)
CAT_COLS = [c for c in CATEGORICAL_VALUES.keys() if c != 'Obesity']

for col in CAT_COLS:
    counts = df[col].value_counts()
    pct = df[col].value_counts(normalize=True).mul(100).round(2)
    freq_df = pd.DataFrame({'Contagem': counts, 'Percentual (%)': pct})
    print(f'\n--- {col} ---')
    print(freq_df.to_string())

## 4. Distribuição das Classes de Obesidade

A variável target `Obesity` possui **7 classes** que representam diferentes níveis de peso corporal. Identificar o desbalanceamento entre essas classes é fundamental para escolher a estratégia de treinamento adequada (ex: `StratifiedShuffleSplit`, métricas macro-F1).

**Classes (em ordem crescente de severidade):**
1. Abaixo do Peso (Insufficient_Weight)
2. Peso Normal (Normal_Weight)
3. Sobrepeso Nível I (Overweight_Level_I)
4. Sobrepeso Nível II (Overweight_Level_II)
5. Obesidade Tipo I (Obesity_Type_I)
6. Obesidade Tipo II (Obesity_Type_II)
7. Obesidade Tipo III (Obesity_Type_III)

**Implicações:** Se houver desbalanceamento significativo, o modelo pode ter viés para as classes majoritárias. A métrica `f1_macro` (média não ponderada do F1 por classe) é mais adequada do que a acurácia simples nesse cenário.

In [ ]:
# Distribuição das classes de obesidade com rótulos PT-BR
obesity_counts = df['Obesity'].value_counts()

# Ordenar pelas 7 classes na ordem canônica
CLASS_ORDER = list(OBESITY_LABELS_PT.keys())
obesity_counts = obesity_counts.reindex(CLASS_ORDER).fillna(0).astype(int)
labels_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]

# Tabela de frequências
obesity_df = pd.DataFrame({
    'Classe (EN)': CLASS_ORDER,
    'Classe (PT-BR)': labels_pt,
    'Contagem': obesity_counts.values,
    'Percentual (%)': (obesity_counts.values / len(df) * 100).round(2)
})
print('=== Distribuição das Classes de Obesidade ===')
print(obesity_df.to_string(index=False))

# Gráfico de barras
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#3B82F6', '#22C55E', '#EAB308', '#F97316', '#EF4444', '#DC2626', '#B91C1C']
bars = ax.bar(labels_pt, obesity_counts.values, color=colors, edgecolor='white', linewidth=0.8)

# Anotações de contagem e percentual
total = len(df)
for bar, count in zip(bars, obesity_counts.values):
    pct = count / total * 100
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        f'{count}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_title('Distribuição das 7 Classes de Obesidade', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Classe de Obesidade', fontsize=12)
ax.set_ylabel('Número de Registros', fontsize=12)
ax.tick_params(axis='x', rotation=30)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

# Análise de desbalanceamento
max_class = obesity_counts.idxmax()
min_class = obesity_counts.idxmin()
ratio = obesity_counts.max() / obesity_counts.min()
print(f'\nClasse mais frequente: {OBESITY_LABELS_PT[max_class]} ({obesity_counts.max()} registros)')
print(f'Classe menos frequente: {OBESITY_LABELS_PT[min_class]} ({obesity_counts.min()} registros)')
print(f'Razão de desbalanceamento: {ratio:.2f}x')

## 5. Histogramas e Box Plots — Features Numéricas

Os histogramas revelam a forma da distribuição de cada feature (simetria, assimetria, multimodalidade). Os box plots agrupados por classe de obesidade permitem identificar quais features têm maior poder discriminativo entre as classes.

**Interpretação dos box plots:**
- A caixa representa o intervalo interquartil (IQR = Q3 − Q1)
- A linha central é a mediana
- Os whiskers cobrem 1.5× o IQR
- Pontos fora dos whiskers são outliers

**Implicações:** Features com distribuições muito diferentes entre classes (ex: Weight, Height) são candidatas a bons preditores. Features com distribuições sobrepostas (ex: TUE) têm menor poder discriminativo isoladamente.

In [ ]:
# Histogramas das features numéricas
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

NOMES_PT = {
    'Age': 'Idade (anos)',
    'Height': 'Altura (m)',
    'Weight': 'Peso (kg)',
    'FCVC': 'Freq. Consumo Vegetais',
    'NCP': 'Nº Refeições Principais',
    'CH2O': 'Consumo de Água (L/dia)',
    'FAF': 'Freq. Atividade Física',
    'TUE': 'Tempo Uso Tecnologia (h)'
}

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    ax.hist(df[col], bins=30, color='#3B82F6', edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='#EF4444', linestyle='--', linewidth=1.5, label=f'Média: {df[col].mean():.2f}')
    ax.axvline(df[col].median(), color='#22C55E', linestyle=':', linewidth=1.5, label=f'Mediana: {df[col].median():.2f}')
    ax.set_title(NOMES_PT[col], fontsize=10, fontweight='bold')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')
    ax.legend(fontsize=8)

fig.suptitle('Histogramas das Features Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots das features numéricas agrupados por classe de obesidade
# Criar coluna com rótulos PT-BR para o eixo x
df['Classe_PT'] = df['Obesity'].map(OBESITY_LABELS_PT)

# Ordem das classes para o eixo x
order_pt = [OBESITY_LABELS_PT[c] for c in CLASS_ORDER]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

palette = dict(zip(order_pt, ['#3B82F6', '#22C55E', '#EAB308', '#F97316', '#EF4444', '#DC2626', '#B91C1C']))

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    sns.boxplot(
        data=df, x='Classe_PT', y=col, order=order_pt,
        palette=palette, ax=ax, linewidth=0.8, fliersize=3
    )
    ax.set_title(NOMES_PT[col], fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Valor')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Box Plots por Classe de Obesidade — Features Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Remover coluna auxiliar
df.drop(columns=['Classe_PT'], inplace=True)

## 6. Correlação de Spearman

A correlação de Spearman é uma medida não-paramétrica de associação monotônica entre variáveis. É mais robusta do que a correlação de Pearson para dados com distribuições não-normais ou relações não-lineares — características comuns neste dataset.

**Metodologia:**
- O target `Obesity` é codificado ordinalmente: Insufficient_Weight=0, Normal_Weight=1, Overweight_Level_I=2, Overweight_Level_II=3, Obesity_Type_I=4, Obesity_Type_II=5, Obesity_Type_III=6
- As features numéricas são correlacionadas diretamente com o target codificado
- As features categóricas binárias (yes/no) são convertidas para 0/1 antes do cálculo
- O heatmap exibe a matriz de correlação completa entre todas as features e o target

**Interpretação:**
- Valores próximos de +1: correlação positiva forte (feature aumenta com o nível de obesidade)
- Valores próximos de -1: correlação negativa forte (feature diminui com o nível de obesidade)
- Valores próximos de 0: pouca ou nenhuma correlação monotônica

**Implicações:** Features com alta correlação absoluta com o target são candidatas a bons preditores. Correlações entre features (multicolinearidade) podem indicar redundância e oportunidades de feature selection.

In [ ]:
# Codificação ordinal do target Obesity
OBESITY_ORDINAL = {
    'Insufficient_Weight': 0,
    'Normal_Weight': 1,
    'Overweight_Level_I': 2,
    'Overweight_Level_II': 3,
    'Obesity_Type_I': 4,
    'Obesity_Type_II': 5,
    'Obesity_Type_III': 6
}

# Preparar DataFrame numérico para correlação
df_corr = df.copy()
df_corr['Obesity_Ordinal'] = df_corr['Obesity'].map(OBESITY_ORDINAL)

# Codificar features binárias yes/no → 1/0
BINARY_COLS = ['family_history', 'FAVC', 'SMOKE', 'SCC']
for col in BINARY_COLS:
    df_corr[col] = (df_corr[col].str.lower() == 'yes').astype(int)

# Codificar Gender: Male=1, Female=0
df_corr['Gender'] = (df_corr['Gender'] == 'Male').astype(int)

# Codificar CAEC e CALC ordinalmente
ORDINAL_MAP = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df_corr['CAEC'] = df_corr['CAEC'].map(ORDINAL_MAP)
df_corr['CALC'] = df_corr['CALC'].map(ORDINAL_MAP)

# Codificar MTRANS ordinalmente (por frequência de uso ativo)
MTRANS_MAP = {'Automobile': 0, 'Motorbike': 1, 'Public_Transportation': 2, 'Bike': 3, 'Walking': 4}
df_corr['MTRANS'] = df_corr['MTRANS'].map(MTRANS_MAP)

# Selecionar colunas para a matriz de correlação
CORR_COLS = NUMERIC_COLS + ['Gender', 'family_history', 'FAVC', 'CAEC', 'SMOKE',
                             'CH2O', 'SCC', 'CALC', 'MTRANS', 'Obesity_Ordinal']
# Remover duplicatas mantendo a ordem
seen = set()
CORR_COLS_UNIQUE = [c for c in CORR_COLS if not (c in seen or seen.add(c))]

# Calcular correlação de Spearman
spearman_matrix = df_corr[CORR_COLS_UNIQUE].corr(method='spearman')

# Rótulos PT-BR para o heatmap
LABEL_MAP = {
    'Age': 'Idade', 'Height': 'Altura', 'Weight': 'Peso',
    'FCVC': 'Freq. Vegetais', 'NCP': 'Nº Refeições', 'CH2O': 'Água (L/dia)',
    'FAF': 'Ativ. Física', 'TUE': 'Uso Tecnologia',
    'Gender': 'Gênero', 'family_history': 'Hist. Familiar',
    'FAVC': 'Alim. Calórica', 'CAEC': 'Comer Fora',
    'SMOKE': 'Tabagismo', 'SCC': 'Monitor Calorias',
    'CALC': 'Álcool', 'MTRANS': 'Transporte',
    'Obesity_Ordinal': 'Obesidade (Target)'
}
labels = [LABEL_MAP.get(c, c) for c in CORR_COLS_UNIQUE]

# Heatmap da correlação de Spearman
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.zeros_like(spearman_matrix, dtype=bool)
# Sem máscara — exibir matriz completa para análise de multicolinearidade

sns.heatmap(
    spearman_matrix,
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 8},
    ax=ax
)

ax.set_title('Matriz de Correlação de Spearman — Features × Target', fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

# Correlações com o target ordenadas por valor absoluto
target_corr = spearman_matrix['Obesity_Ordinal'].drop('Obesity_Ordinal').abs().sort_values(ascending=False)
target_corr_signed = spearman_matrix['Obesity_Ordinal'].drop('Obesity_Ordinal').reindex(target_corr.index)

corr_df = pd.DataFrame({
    'Feature': [LABEL_MAP.get(c, c) for c in target_corr.index],
    'Correlação de Spearman': target_corr_signed.values.round(4),
    '|Correlação|': target_corr.values.round(4)
})
print('\n=== Correlação de Spearman com o Target (Obesity) ===')
print(corr_df.to_string(index=False))

In [ ]:
# Gráfico de barras horizontais — correlação com o target
fig, ax = plt.subplots(figsize=(10, 6))

colors_bar = ['#EF4444' if v > 0 else '#3B82F6' for v in target_corr_signed.values]
bars = ax.barh(
    [LABEL_MAP.get(c, c) for c in target_corr_signed.index],
    target_corr_signed.values,
    color=colors_bar, edgecolor='white', linewidth=0.5
)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Correlação de Spearman com Obesidade (Target)', fontsize=11)
ax.set_title('Correlação de Spearman: Features × Target (Obesity)', fontsize=13, fontweight='bold')

# Anotações
for bar, val in zip(bars, target_corr_signed.values):
    offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.show()

## 7. Conclusões da EDA

### Principais achados

**1. Qualidade dos dados**
- O dataset apresenta boa qualidade após a limpeza com `DataCleaner`: sem valores ausentes e poucas ou nenhuma duplicata.
- Todas as 17 colunas esperadas estão presentes e os valores estão dentro dos ranges definidos em `src/config.py`.

**2. Distribuição das classes (desbalanceamento)**
- O dataset apresenta desbalanceamento moderado entre as 7 classes de obesidade.
- A estratégia de divisão com `StratifiedShuffleSplit` é essencial para preservar as proporções em treino/validação/teste.
- A métrica de otimização `f1_macro` é mais adequada do que a acurácia simples para avaliar o modelo neste cenário.

**3. Features mais discriminativas (correlação de Spearman)**
- **Peso (Weight)** e **Altura (Height)** apresentam as maiores correlações absolutas com o target, o que é esperado dado que o IMC (BMI = Weight/Height²) é o principal indicador clínico de obesidade.
- **Histórico familiar (family_history)** e **consumo de alimentos calóricos (FAVC)** também mostram correlação positiva relevante.
- **Frequência de atividade física (FAF)** apresenta correlação negativa — maior atividade física está associada a menores níveis de obesidade.
- Features como **tabagismo (SMOKE)** e **monitoramento de calorias (SCC)** têm correlação baixa, indicando menor poder preditivo isolado.

**4. Distribuições das features numéricas**
- **Age** e **Weight** apresentam distribuições assimétricas positivas.
- **FCVC, NCP, CH2O, FAF, TUE** são variáveis discretizadas (escala 0–3 ou 1–4), com distribuições concentradas em poucos valores.
- Os box plots revelam que **Weight** e **Height** têm separação clara entre as classes de obesidade, confirmando seu alto poder discriminativo.

**5. Implicações para o pipeline de ML**
- A feature engineered **BMI = Weight / Height²** deve capturar a relação entre peso e altura de forma mais informativa do que as duas features separadas.
- O `eating_score` (combinação de FAVC, FCVC, NCP, CAEC) e o `lifestyle_score` (FAF, TUE, CALC, SMOKE) devem agregar informação complementar.
- O `StandardScaler` em Age, Height, Weight, BMI e o `MinMaxScaler` em FCVC, NCP, CH2O, FAF, TUE são adequados dado o perfil das distribuições observadas.

---
*Próximo passo: `02_feature_engineering.ipynb` — demonstração das transformações do `FeatureEngineer` com justificativas visuais.*